# Lab 2 — Getting started with Qrisp on IQM

**Workshop block 4 · 30 min** · adapted from IQM Academy, *Get started with Qrisp on IQM*
(https://www.iqmacademy.com/learn/qrisp/00-intro/) and the Qrisp tutorial (https://qrisp.eu).

Qrisp is a high-level quantum programming framework built around **quantum variables** instead of qubits and
gates, and it is the default SDK on IQM Resonance. If you write C++/Fortran for HPC, think of it as moving
from hand-written assembly to typed variables, operators and a compiler that manages registers for you.

In this notebook you will …

* … get started with Qrisp: `QuantumVariable`, `QuantumFloat`, automatic uncomputation,
* … run a Qrisp program on a simulator, an IQM **mock** backend and a **real** IQM QPU,
* … run two hybrid algorithms (Grover search, QAOA MaxCut) and look at their resource cost,
* … send the *same* Qrisp program through **QDMI**, connecting back to Lab 1.

## Installing the necessary packages
Make sure that you install Qrisp with the `iqm` extra. For the QDMI bridge at the end you also need the Lab 1 environment.

In [ ]:
# %pip install -q "qrisp[iqm]" networkx matplotlib
# On some shells you need quotes around the extras:  pip install "qrisp[iqm]"

## Imports
`from qrisp import *` is not best practice, but it keeps the notebook short.

In [ ]:
from qrisp import *
from qrisp.interface import IQMBackend
import numpy as np, matplotlib.pyplot as plt

## 1 · Quantum variables, not qubits

A `QuantumVariable` hides qubit management, gives human-readable inputs and outputs and supports typed
arithmetic. Gate functions still work on it when you want low-level control.

In [ ]:
qv = QuantumVariable(3)
h(qv[0])
cx(qv[0], qv[1])
cx(qv[0], qv[2])          # a 3-qubit GHZ state
print(qv.qs)              # the QuantumSession holds the circuit
print(qv)                 # printing a variable = simulate + measure

**Typed data.** `QuantumFloat(msize, exponent, signed)` is a fixed-point number. As in the IQM Academy example we
encode a `2`, multiply it with itself and measure. Qrisp synthesises the multiplier for us.

In [ ]:
a = QuantumFloat(2)
a[:] = 2
print("a          ->", a.get_measurement())
b = a * a
print("a * a      ->", b.get_measurement())
print("qubits used:", b.qs.compile().num_qubits(), "| depth:", b.qs.compile().depth())

## 2 · Running on IQM Resonance

### Token authentication
To access IQM quantum computers you need an API token: **https://resonance.meetiqm.com** → *Create Token* in the
side panel. As in the IQM Academy notebook we cache it in `token.txt` next to the notebook (add it to `.gitignore`).

In [ ]:
import os
from pathlib import Path
from getpass import getpass

token_file = Path("token.txt")
token = os.environ.get("RESONANCE_API_KEY") or os.environ.get("IQM_TOKEN")
if not token:
    if not token_file.exists():
        t = ""
        while t == "":
            t = getpass("Please enter the token you generated on IQM Resonance: ")
        token_file.write_text(t)
    else:
        print("Reusing existing token.txt")
    token = token_file.read_text().strip()

### Backends: mock first, then real hardware
`device_instance="<qpu>:mock"` gives a backend with the real device's interface and a noise model, for **zero credits**.
Always test there first. The `IQMBackend` keyword for the token changed between Qrisp releases (`api_token=` →
`token=`); the helper accepts both.

In [ ]:
QPU = os.environ.get("IQM_QC_ALIAS", "garnet")   # garnet | emerald | sirius ... (what your account can access)

def make_iqm_backend(instance):
    for kw in ({"token": token}, {"api_token": token}):
        try:
            return IQMBackend(device_instance=instance, **kw)
        except TypeError:
            continue
    raise RuntimeError("Could not construct IQMBackend; check your qrisp[iqm] version")

iqm_mock = make_iqm_backend(f"{QPU}:mock")
iqm_real = make_iqm_backend(QPU)

### Compare all three runs: simulator vs. mock vs. real hardware
The real run consumes credits.

In [ ]:
def squared_two():
    a = QuantumFloat(2); a[:] = 2
    return a * a

sim_result = squared_two().get_measurement()
print(f"Qrisp simulator : {sim_result}   (noiseless)")

mock_result = squared_two().get_measurement(backend=iqm_mock, shots=500)
print(f"IQM mock        : {mock_result}   (noise model, zero credits)")

In [ ]:
RUN_ON_HARDWARE = False      # flip when you're ready to spend credits
if RUN_ON_HARDWARE:
    real_result = squared_two().get_measurement(backend=iqm_real, shots=500)
    print(f"IQM {QPU} (real): {real_result}")

**Tip.** The arithmetic circuit is deep. Hardware noise spreads the result over wrong values, which is why
shallow, structure-aware algorithms (and later error correction) matter. Compare with the 3-qubit GHZ:

In [ ]:
def ghz(n=3):
    q = QuantumVariable(n); h(q[0])
    for i in range(1, n): cx(q[0], q[i])
    return q

print("GHZ on mock:", ghz().get_measurement(backend=iqm_mock, shots=500))

## 3 · Algorithm 1: solve $x^2 = 0.25$ with Grover's algorithm

The oracle squares a `QuantumFloat`, compares with 0.25 and phase-flips the match. `@auto_uncompute` cleans up
every temporary (the product and the comparison bit) automatically, something you would hand-code in other SDKs.

In [ ]:
from math import pi
from qrisp.grover import diffuser

@auto_uncompute
def sqrt_oracle(qf):
    temp_qbool = qf * qf == 0.25
    z(temp_qbool)

qf = QuantumFloat(3, -1, signed=True)
n = qf.size
iterations = int(0.25 * pi * (2 ** n / 2) ** 0.5)   # two solutions expected: +0.5 and -0.5
h(qf)
for _ in range(iterations):
    sqrt_oracle(qf)
    diffuser(qf)

res = qf.get_measurement()
print({k: round(v, 3) for k, v in sorted(res.items(), key=lambda kv: -kv[1])[:4]})
compiled = qf.qs.compile()
print(f"compiled: {compiled.num_qubits()} qubits (ancillae recycled), depth {compiled.depth()}, ops {compiled.count_ops()}")

## 4 · Algorithm 2: QAOA MaxCut, a hybrid quantum–classical loop

QAOA alternates a cost layer and a mixer layer; a **classical optimiser** tunes the angles from measured energies.
That optimiser loop is exactly the classical↔quantum round trip you timed in Lab 1.
(For production QAOA on IQM hardware, see IQM's `iqm-qaoa` library: https://docs.meetiqm.com/iqm-qaoa/.)

In [ ]:
import networkx as nx
from qrisp.qaoa import QAOAProblem, RX_mixer, create_maxcut_cost_operator, create_maxcut_cl_cost_function

G = nx.Graph([(0, 3), (0, 4), (1, 3), (1, 4), (2, 3), (2, 4)])
cl_cost = create_maxcut_cl_cost_function(G)
maxcut = QAOAProblem(create_maxcut_cost_operator(G), RX_mixer, cl_cost)

qarg = QuantumVariable(G.number_of_nodes())
res = maxcut.run(qarg, depth=3, max_iter=50)          # local simulator
best = sorted(res, key=res.get, reverse=True)[:5]
for cut in best:
    print(f"{cut}  p={res[cut]:.3f}  cut size={-cl_cost({cut: 1})}")

nx.draw(G, with_labels=True, node_color=["#21b5a8" if c == "1" else "#f08a3c" for c in best[0]], font_color="white")
plt.title(f"best cut {best[0]}"); plt.show()

Run the same QAOA with measurements on the IQM mock backend. Each optimiser step submits circuits to Resonance.

In [ ]:
qarg = QuantumVariable(G.number_of_nodes())
res_mock = maxcut.run(qarg, depth=1, max_iter=15, mes_kwargs={"backend": iqm_mock, "shots": 500})
best_mock = max(res_mock, key=res_mock.get)
print("most likely cut on mock:", best_mock, "| cut size", -cl_cost({best_mock: 1}))

## 5 · The bridge to QDMI: same Qrisp program, vendor-neutral device interface

Qrisp compiles to a circuit; QDMI executes circuits. Wrapping a QDMI backend as a Qrisp `VirtualBackend` lets
*any* QDMI device (local DDSIM, Resonance, an on-prem Radiance behind Slurm) run Qrisp code unchanged.
Requires the Lab 1 environment (`mqt-core[qiskit]`, `iqm-qdmi`).

In [ ]:
try:
    from qiskit import transpile
    from qrisp.interface import VirtualBackend
    from mqt.core.plugins.qiskit import QDMIBackend

    def qdmi_qrisp_backend(device_id, **session):
        qdmi = QDMIBackend.from_device_id(device_id, **session)
        def run_func(qc, shots, token=""):
            qk = transpile(qc.to_qiskit(), backend=qdmi, optimization_level=1)
            counts = qdmi.run(qk, shots=shots or 1000).result().get_counts()
            return {k.replace(" ", ""): v for k, v in counts.items()}
        return VirtualBackend(run_func)

    ddsim_via_qdmi = qdmi_qrisp_backend("mqt.ddsim.default")
    print("GHZ via QDMI/DDSIM     :", ghz().get_measurement(backend=ddsim_via_qdmi, shots=500))
    def grover_x2():
        q = QuantumFloat(3, -1, signed=True); h(q)
        for _ in range(iterations):
            sqrt_oracle(q); diffuser(q)
        return q
    top = sorted(grover_x2().get_measurement(backend=ddsim_via_qdmi, shots=1000).items(), key=lambda kv: -kv[1])[:3]
    print("Grover x^2=0.25 via QDMI:", {k: round(v, 2) for k, v in top})

    from mqt.core.qdmi import builtin_driver
    iqm_ids = [d for d in builtin_driver.registered_device_ids() if d.lower().startswith("iqm")]
    if iqm_ids:
        iqm_via_qdmi = qdmi_qrisp_backend(next((d for d in iqm_ids if "mock" in d), iqm_ids[0]), token=token)
        print("GHZ via QDMI/IQM       :", ghz().get_measurement(backend=iqm_via_qdmi, shots=500))
except ImportError as exc:
    print("QDMI bridge skipped (install the Lab 1 environment):", exc)

## Where to go next
* IQM Academy: Qrisp modules, error reduction, **QEC memory experiment with Qrisp on IQM Emerald**: https://www.iqmacademy.com
* Qrisp tutorials: Shor, QAOA, Jasp (hybrid real-time programs), fault-tolerant compilation: https://qrisp.eu/general/tutorial/
* Lab 3: QSCI, where the quantum part is small and the classical part is an HPC (GPU) job.